# Target shape — `total_points` (Y)

*Read-only informative artifact. This notebook characterises the target so a
human can decide what matters. It produces no gate decisions and no
PROCEED/STOP verdict.*

## Questions a manager asks of the target

- **How many points does each position typically score in a week they play?**
  (a midfielder vs a goalkeeper — what is a "normal" haul-free return?)
- **How often does a starter blank?** How often do they return something
  useful (4+, 6+, 8+ points)?
- **What is the ceiling?** How often does each position deliver a genuine haul
  (11+, 16+ points) that wins a gameweek?
- **How much of a player's scoring is who they are vs which week it is?**
  (deferred — see the variance-decomposition section)

Everything below is **season-pooled** over the study range. Week-to-week
behaviour (streaks, regression, momentum) is deferred to the `temporal/` layer.

## Setup

Load the mart, restrict to the **whole season** (GW 1 to the latest completed
GW) and to the **participation** population (`minutes > 0`), and build position
cohorts.

This is a *descriptive characterisation* notebook, so it uses the full season:
there is no reason to drop early gameweeks. (The GW-6 lower bound in the older
EDA-1 record was a *predictive-evaluation* choice — dropping early-season noise
for cleaner correlations — which does not apply here.)

The population is everyone who **actually featured**: available players with
`minutes > 0`. This is a **participation** filter (the player appeared), **not
a performance gate**. `minutes` can be NULL for some rows; `minutes > 0`
naturally excludes those (NULL comparisons are False). The 60-minute
performance-boundary question — whether `total_points` below it reflects
non-participation rather than performance — is **deferred to the `population/`
layer**, which is the layer meant to study and justify that boundary. Baking it
in here would pre-judge exactly that question.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from dal.pipeline import load as load_mart, run as run_pipeline
from dal.exceptions import MartNotBuiltError, MartSchemaError
from research.kernels.descriptive.distribution import (
    compute_distribution_stats,
    compare_cohorts,
)

try:
    _result = load_mart()
except (MartNotBuiltError, MartSchemaError) as _e:
    print(f"Rebuilding mart ({type(_e).__name__})...")
    run_pipeline(force=True)
    _result = load_mart()

# Descriptive characterisation uses the WHOLE season: GW 1 to the latest
# completed GW. No early-GW lower bound (that was a predictive-evaluation
# choice in the older EDA-1 record, not relevant here).
STUDY_GW_MIN = 1
STUDY_GW_MAX = _result.data_cutoff_gw

# Analytical population: PARTICIPATION filter, not a performance gate.
# Available players who actually featured -> minutes > 0. `minutes` can be NULL
# for some rows; minutes > 0 naturally excludes NULLs (NULL comparisons are
# False), stated explicitly below. The 60-minute performance boundary is NOT
# imposed here -- that question is deferred to the population/ layer.
mart = _result.mart
df = mart[mart["gw"].between(STUDY_GW_MIN, STUDY_GW_MAX)].copy()
df = df[df["minutes"].notna() & (df["minutes"] > 0)].copy()

POSITIONS = ["GK", "DEF", "MID", "FWD"]
cohorts = {pos: df[df.position == pos] for pos in POSITIONS}

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", "{:.3f}".format)

print(f"Study range: GW {STUDY_GW_MIN} - GW {STUDY_GW_MAX} (whole season, from mart data_cutoff_gw)")
print(f"Population: minutes > 0 (participation, not a performance gate), n = {len(df):,} player-gameweeks")
for pos in POSITIONS:
    print(f"  {pos}: {len(cohorts[pos]):>6,}")


## (a) Distribution of `total_points` by position

**What we measure** — the full univariate shape of `total_points` (count,
mean, median, std, percentiles, skew, kurtosis) within each position cohort,
via `compare_cohorts`.

**What it means** — the typical return and spread a manager should expect from
a *starting* player at each position. Median is the "coin-flip" return; the
p90/p99 columns show how far the upper tail reaches. Use this to calibrate
expectations when picking a squad slot — a DEF and a MID are not competing on
the same scale.

**What it doesn't mean** — these are **season-pooled** statistics across all
players and all weeks in range. A single player's week-to-week distribution is
narrower and is deferred to the `temporal/` layer. The mean is not a forecast
for any individual gameweek; right-skew means the mean sits above the median
and is pulled by rare hauls.

**Guiding question** — *How many points does each position typically score in
a week they play, and what is the spread?*

In [ ]:
dist_by_pos = compare_cohorts(cohorts, value_col="total_points")
display(
    dist_by_pos[
        ["count", "mean", "median", "std", "p75", "p90", "p99", "max", "skew", "kurtosis"]
    ].round(2)
)


In [ ]:
# Distribution shape is what a table hides here: overlaid histograms show
# where each position's mass sits and how far the right tail runs.
colours = {"GK": "#9467bd", "DEF": "#1f77b4", "MID": "#2ca02c", "FWD": "#d62728"}
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6), sharey=True)
x_max = int(df["total_points"].max())
bins = range(int(df["total_points"].min()), x_max + 2)
for ax, pos in zip(axes, POSITIONS):
    s = cohorts[pos]["total_points"].dropna()
    ax.hist(s, bins=bins, color=colours[pos], edgecolor="white", linewidth=0.3)
    ax.axvline(s.median(), color="black", linestyle="--", linewidth=1.0)
    ax.set_title(f"{pos}  (median={s.median():.0f})", fontsize=11)
    ax.set_xlabel("total_points")
axes[0].set_ylabel("count")
fig.suptitle("total_points by position (minutes > 0, season-pooled)", y=1.02)
plt.tight_layout()
plt.show()


## (b) Consistency — blank rate and return rate by position

**What we measure** — for each position cohort, the share of player-gameweeks
that are a **blank** (`total_points == 0`) and the share that clear successive
**return** thresholds (`>= 4`, `>= 6`, `>= 8`). Computed inline as simple
proportions.

**What it means** — consistency is a transfer/captaincy property: a player who
clears 4+ most weeks is a reliable hold; one who frequently blanks is a
gamble. The 4 / 6 / 8 ladder maps to "did something", "good week", "great
week". GK and DEF returns are clean-sheet-driven, so their blank behaviour
differs structurally from attackers.

**What it doesn't mean** — these are **season-pooled** rates, not the
probability that a specific player blanks next week. A 30% pooled blank rate
mixes reliable and volatile players; it does not describe one player's
week-to-week reliability, which is deferred to the `temporal/` layer. A blank
here also means exactly zero points — it does not capture "played but
underwhelming" (1-3 pt) weeks, which the threshold ladder shows separately.

This cohort is the **participation** population (`minutes > 0` — the player
appeared), **not a performance-gated** one. A short cameo counts as
participation, so the blank rate here includes brief appearances that returned
nothing. Whether a 60-minute performance boundary should be applied — and what
it would do to these rates — is **deferred to the `population/` layer**, not
pre-judged here.

**Guiding question** — *How often does a starter blank, and how often do they
return something useful?*

In [ ]:
rows = []
for pos in POSITIONS:
    y = cohorts[pos]["total_points"].dropna()
    n = len(y)
    rows.append({
        "position": pos,
        "n": n,
        "blank_rate_%": round((y == 0).mean() * 100, 1),
        "return_4+_%": round((y >= 4).mean() * 100, 1),
        "return_6+_%": round((y >= 6).mean() * 100, 1),
        "return_8+_%": round((y >= 8).mean() * 100, 1),
    })
consistency = pd.DataFrame(rows)
display(consistency)


## (c) Haul frequency by position

**What we measure** — the share of player-gameweeks exceeding haul thresholds
(`> 10` and `> 15` points) within each position cohort. Computed inline.

**What it means** — hauls are the gameweek-winning, captaincy-justifying
events. This shows how rare they are and which positions produce them. A
position with a thicker `> 10` mass is where captaincy upside concentrates.

**What it doesn't mean** — this is the **season-pooled** rate of haul
*occurrence*, not the probability that a given player hauls in a given week,
and not which signal predicts it. It says nothing about whether a haul is
forecastable — only how frequent it is. Predictive value of any signal is out
of scope here; timing is deferred to the `temporal/` layer.

**Guiding question** — *What is the ceiling — how often does each position
deliver a genuine haul?*

In [ ]:
rows = []
for pos in POSITIONS:
    y = cohorts[pos]["total_points"].dropna()
    rows.append({
        "position": pos,
        "n": len(y),
        "haul_>10_%": round((y > 10).mean() * 100, 2),
        "haul_>15_%": round((y > 15).mean() * 100, 2),
    })
hauls = pd.DataFrame(rows)
display(hauls)


## DEFERRED — pending kernel: variance decomposition (between-player vs within-player)

This section will split total `total_points` variance into a **between-player**
component (persistent player quality — who they are) and a **within-player**
component (week-to-week event variation — which week it is), per position. It
answers whether a player's scoring is dominated by their identity or by the
gameweek, which determines whether a correlated signal is sorting players or
timing returns.

It requires a panel variance-decomposition kernel (one-way ANOVA SS split,
weighted by group size, guaranteeing `SS_between + SS_within = SS_total`) that
is not yet built in `research.kernels`. No code is run here until that kernel
exists; the SS-based decomposition belongs in a kernel, not inline in an
informative notebook.

## What the target looks like

Plain-language summary of the descriptive picture (not a verdict):

- Every position's `total_points` is **right-skewed** — the median sits below
  the mean, and a thin upper tail of hauls pulls the average up.
- **GK** has the lowest mean and the most compressed tail; returns are
  clean-sheet-bound. **DEF** carries elevated low-score mass for the same
  structural reason. **MID** and **FWD** carry the heaviest right tails.
- **Blanks** (exactly zero) appear at a higher rate in this `minutes > 0`
  **participation** population than they would under a 60-minute performance
  gate, because brief cameos that returned nothing are included here by design.
  Useful returns (4+) and great weeks (8+) thin out sharply as the threshold
  rises.
- **Hauls** (>10, >15) are rare everywhere and concentrate in attacking
  positions.

All figures are **whole-season**, season-pooled, over the **participation**
population (`minutes > 0`, not a performance gate). The 60-minute boundary is
left to the `population/` layer; whether these returns are timeable
week-to-week is left to the `temporal/` layer; the between/within split is
deferred above.